##### Marketing raw files inspection from S3.

In [0]:
display(
    dbutils.fs.ls(
        "s3://voyage-data-plateform-yuvaraj-india/raw/airbyte/voyage_raw/"
    )
)

In [0]:
display(
    dbutils.fs.ls(
        "s3://voyage-data-plateform-yuvaraj-india/raw/airbyte/voyage_raw/campaigns/"
    )
)

In [0]:
base_path = "s3://voyage-data-plateform-yuvaraj-india/raw/airbyte/voyage_raw/"

marketing_folders = [
    "campaigns",
    "ad_groups",
    "ads",
    "campaign_performance",
]

for folder in marketing_folders:
    path = f"{base_path}{folder}/"
    print(f"\n===== {folder} =====")
    files = dbutils.fs.ls(path)
    display(files)

In [0]:
for folder in marketing_folders:
    path = f"{base_path}{folder}/"

    print(f"\n===== {folder} =====")

    df = spark.read.json(path)

    print("Row count:", df.count())

    df.printSchema()

    display(df.limit(3))

In [0]:
from pyspark.sql import functions as F


BASE_PATH = (
    "s3://voyage-data-plateform-yuvaraj-india/"
    "raw/airbyte/voyage_raw/"
)

MARKETING_STREAMS = {
    "campaigns": "marketing_campaigns",
    "ad_groups": "marketing_ad_groups",
    "ads": "marketing_ads",
    "campaign_performance": "marketing_campaign_performance",
}


def read_airbyte_stream(stream_name: str):
    path = f"{BASE_PATH}{stream_name}/"

    raw_df = spark.read.json(path)

    flattened_df = raw_df.select(
        "_airbyte_data.*",
        "_airbyte_extracted_at",
        "_airbyte_generation_id",
        "_airbyte_meta",
        "_airbyte_raw_id",
    )

    return flattened_df


for stream_name, table_name in MARKETING_STREAMS.items():

    print(f"\n===== {stream_name} =====")

    df = read_airbyte_stream(stream_name)

    print(f"Raw Bronze rows: {df.count()}")
    print(f"Writing: voyage.bronze.{table_name}")

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"voyage.bronze.{table_name}")
    )

    print(f"Completed: voyage.bronze.{table_name}")

In [0]:
for stream_name, table_name in MARKETING_STREAMS.items():

    table = f"voyage.bronze.{table_name}"

    df = spark.table(table)

    print(f"\n===== {table} =====")
    print("Rows:", df.count())

    if stream_name == "campaigns":
        print("Distinct campaigns:", df.select("campaign_id").distinct().count())

    elif stream_name == "ad_groups":
        print("Distinct ad groups:", df.select("ad_group_id").distinct().count())

    elif stream_name == "ads":
        print("Distinct ads:", df.select("ad_id").distinct().count())

    elif stream_name == "campaign_performance":
        print(
            "Distinct metrics:",
            df.select("metric_id").distinct().count()
        )